In [1]:
import sys
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@17/17.0.19/libexec/openjdk.jdk/Contents/Home"

# Ensure PySpark uses the correct Python executable from your uv venv
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
import pyspark
from pyspark.sql import SparkSession

In [7]:
SPARK_VERSION = "3.5"
spark = SparkSession.builder.appName(
    "SensorAnomalyDetection"
).config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_2.12:{SPARK_VERSION}.0").getOrCreate()

26/06/05 10:48:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [8]:
BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:29092")
INPUT_TOPIC = os.getenv("SENSOR_TOPIC", "site-sensor-raw")
OUTPUT_TOPIC = os.getenv("ALERTS_TOPIC", "alerts")
ALGORITHM = os.getenv("ALGORITHM", "Thresholding") 
PYSPARK_SUBMIT_ARGS = os.getenv("PYSPARK_SUBMIT_ARGS")
PYSPARK_SUBMIT_ARGS

'--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1 pyspark-shell'

In [9]:
from pyspark.sql.functions import col, from_json
from system.schemas import SENSOR_SCHEMA

df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", INPUT_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

parsed = (
    df.selectExpr("CAST(value AS STRING)")
    .select(from_json(col("value"), SENSOR_SCHEMA).alias("data"))
    .select("data.*")
    .withColumn("event_time", col("timestamp").cast("timestamp"))
)

In [10]:
query = (
    parsed.writeStream
    .format("csv")
    .option("path", "./output-csv-data")
    .option("checkpointLocation", "./checkpoint")
    .start()
)

has_finished = query.awaitTermination(timeout=60)

if not has_finished:
    print("Time limit reached. Shutting down the stream gracefully...")
    query.stop()  # Gracefully stops the background streaming thread

26/06/05 10:48:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/05 10:48:59 WARN ClientUtils: Couldn't resolve server kafka:29092 from bootstrap.servers as DNS resolution failed for kafka
26/06/05 10:48:59 WARN KafkaOffsetReaderAdmin: Error in attempt 1 getting Kafka offsets: 
org.apache.kafka.common.KafkaException: Failed to create new KafkaAdminClient
	at org.apache.kafka.clients.admin.KafkaAdminClient.createInternal(KafkaAdminClient.java:561)
	at org.apache.kafka.clients.admin.Admin.create(Admin.java:147)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.createAdmin(ConsumerStrategy.scala:49)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.createAdmin$(ConsumerStrategy.scala:46)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.createAdmin(ConsumerStrategy.scala:102)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.admin(KafkaOffsetReaderAdmin.scala:69)
	at org.apache.spark.sql.kafka0

StreamingQueryException: [STREAM_FAILED] Query [id = e3d374b0-68e2-45ef-9482-85303e184fbf, runId = 4f9e61b4-eb7b-42d8-9afd-ad3c865fac5d] terminated with exception: Failed to create new KafkaAdminClient SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = e3d374b0-68e2-45ef-9482-85303e184fbf, runId = 4f9e61b4-eb7b-42d8-9afd-ad3c865fac5d]
Current Committed Offsets: {}
Current Available Offsets: {}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSourceV1 FileSink[file:/Users/sanyamsuniljain/Desktop/NoDust/output-csv-data], e3d374b0-68e2-45ef-9482-85303e184fbf, [path=./output-csv-data, checkpointLocation=./checkpoint], Append
+- ~Project [sensor_id#46, site_id#47, timestamp#48, pollutant_type#49, concentration#50, unit#51, cast(timestamp#48 as timestamp) AS event_time#52]
   +- ~Project [data#45.sensor_id AS sensor_id#46, data#45.site_id AS site_id#47, data#45.timestamp AS timestamp#48, data#45.pollutant_type AS pollutant_type#49, data#45.concentrati